# Phishing URL Detection — Preprocessing Pipeline
**Dataset**: PHIUSIIL Phishing URL Dataset  
**Goal**: Clean, validate, and select features for XGBoost classification  
**Where**: Class Labels -->
Label 1 corresponds to a legitimate URL, label 0 to a phishing URL

## Pipeline Steps
1. Data loading & inspection
2. Drop non-feature columns
3. Duplicate column detection
4. Train/test split (80/20, stratified)
5. Outlier detection & capping (IQR, train-only)
6. Feature selection — Correlation → VIF → SelectFromModel (RF)
7. Artifact saving for inference pipeline

In [58]:
import pandas as pd 
import os 
import hashlib
import numpy as np 
from sklearn.model_selection import train_test_split,cross_val_score, StratifiedKFold
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import json
import joblib

In [4]:
phiusiil_df = pd.read_csv("dataset/raw_phiusiil.csv")

In [5]:
phiusiil_df.shape

(235795, 56)

In [6]:
phiusiil_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 235795 entries, 0 to 235794
Data columns (total 56 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   FILENAME                    235795 non-null  str    
 1   URL                         235795 non-null  str    
 2   URLLength                   235795 non-null  int64  
 3   Domain                      235795 non-null  str    
 4   DomainLength                235795 non-null  int64  
 5   IsDomainIP                  235795 non-null  int64  
 6   TLD                         235795 non-null  str    
 7   URLSimilarityIndex          235795 non-null  float64
 8   CharContinuationRate        235795 non-null  float64
 9   TLDLegitimateProb           235795 non-null  float64
 10  URLCharProb                 235795 non-null  float64
 11  TLDLength                   235795 non-null  int64  
 12  NoOfSubDomain               235795 non-null  int64  
 13  HasObfuscation           

In [7]:
phiusiil_df.isnull().sum()

FILENAME                      0
URL                           0
URLLength                     0
Domain                        0
DomainLength                  0
IsDomainIP                    0
TLD                           0
URLSimilarityIndex            0
CharContinuationRate          0
TLDLegitimateProb             0
URLCharProb                   0
TLDLength                     0
NoOfSubDomain                 0
HasObfuscation                0
NoOfObfuscatedChar            0
ObfuscationRatio              0
NoOfLettersInURL              0
LetterRatioInURL              0
NoOfDegitsInURL               0
DegitRatioInURL               0
NoOfEqualsInURL               0
NoOfQMarkInURL                0
NoOfAmpersandInURL            0
NoOfOtherSpecialCharsInURL    0
SpacialCharRatioInURL         0
IsHTTPS                       0
LineOfCode                    0
LargestLineLength             0
HasTitle                      0
Title                         0
DomainTitleMatchScore         0
URLTitle

In [8]:
phiusiil_df.drop(columns=['FILENAME','Title', 'Domain', 'URL', 'TLD'], inplace=True)

In [9]:
phiusiil_df.head()

,URLLength,DomainLength,IsDomainIP,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,31,24,0,100.0,1.000000,0.522907,0.061933,3,1,0,...,0,0,1,34,20,28,119,0,124,1
1,23,16,0,100.0,0.666667,0.032650,0.050207,2,1,0,...,0,0,1,50,9,8,39,0,217,1
2,29,22,0,100.0,0.866667,0.028555,0.064129,2,2,0,...,0,0,1,10,2,7,42,2,5,1
3,26,19,0,100.0,1.000000,0.522907,0.057606,3,1,0,...,1,1,1,3,27,15,22,1,31,1
4,33,26,0,100.0,1.000000,0.079963,0.059441,3,1,0,...,1,0,1,244,15,34,72,1,85,1


### Function for finding duplicate identical columns

In [ ]:
def find_duplicate_columns(df: pd.DataFrame) -> dict:

    # hashing every column in one pass
    col_hashes={}
    for col in df.columns:
        col_hash = hashlib.md5(pd.util.hash_array(df[col].values).tobytes()).hexdigest()
        col_hashes[col] = col_hash

    # Group columns by hash value 
    hash_groups = {}
    for col, hash_val in col_hashes.items():
        if hash_val not in hash_groups:
            hash_groups[hash_val] = []
        hash_groups[hash_val].append(col)

    # comparision 
    duplicate_map = {}
    for hash_val, cols in hash_groups.items():
        if len(cols) < 2:
            continue

        original = cols[0]
        for candidate in cols[1:]:
            if df[original].equals(df[candidate]):
                duplicate_map[candidate] = original

    return duplicate_map

In [11]:
find_duplicate_columns(phiusiil_df)

{}

In [12]:
phiusiil_df.head()

,URLLength,DomainLength,IsDomainIP,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,31,24,0,100.0,1.000000,0.522907,0.061933,3,1,0,...,0,0,1,34,20,28,119,0,124,1
1,23,16,0,100.0,0.666667,0.032650,0.050207,2,1,0,...,0,0,1,50,9,8,39,0,217,1
2,29,22,0,100.0,0.866667,0.028555,0.064129,2,2,0,...,0,0,1,10,2,7,42,2,5,1
3,26,19,0,100.0,1.000000,0.522907,0.057606,3,1,0,...,1,1,1,3,27,15,22,1,31,1
4,33,26,0,100.0,1.000000,0.079963,0.059441,3,1,0,...,1,0,1,244,15,34,72,1,85,1


In [13]:
phiusiil_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 235795 entries, 0 to 235794
Data columns (total 51 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   URLLength                   235795 non-null  int64  
 1   DomainLength                235795 non-null  int64  
 2   IsDomainIP                  235795 non-null  int64  
 3   URLSimilarityIndex          235795 non-null  float64
 4   CharContinuationRate        235795 non-null  float64
 5   TLDLegitimateProb           235795 non-null  float64
 6   URLCharProb                 235795 non-null  float64
 7   TLDLength                   235795 non-null  int64  
 8   NoOfSubDomain               235795 non-null  int64  
 9   HasObfuscation              235795 non-null  int64  
 10  NoOfObfuscatedChar          235795 non-null  int64  
 11  ObfuscationRatio            235795 non-null  float64
 12  NoOfLettersInURL            235795 non-null  int64  
 13  LetterRatioInURL         

### Train test split 

In [14]:
X = phiusiil_df.drop(columns='label')
y = phiusiil_df['label']

In [15]:
X.head()

,URLLength,DomainLength,IsDomainIP,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,...,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef
0,31,24,0,100.0,1.000000,0.522907,0.061933,3,1,0,...,1,0,0,1,34,20,28,119,0,124
1,23,16,0,100.0,0.666667,0.032650,0.050207,2,1,0,...,0,0,0,1,50,9,8,39,0,217
2,29,22,0,100.0,0.866667,0.028555,0.064129,2,2,0,...,0,0,0,1,10,2,7,42,2,5
3,26,19,0,100.0,1.000000,0.522907,0.057606,3,1,0,...,0,1,1,1,3,27,15,22,1,31
4,33,26,0,100.0,1.000000,0.079963,0.059441,3,1,0,...,1,1,0,1,244,15,34,72,1,85


In [16]:
y

0         1
1         1
2         1
3         1
4         1
         ..
235790    1
235791    1
235792    1
235793    0
235794    1
Name: label, Length: 235795, dtype: int64

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
X_train.head()

,URLLength,DomainLength,IsDomainIP,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,...,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef
125096,33,25,0,29.649226,0.772727,0.005977,0.056982,2,2,0,...,0,0,0,0,0,0,0,0,0,2
52769,19,13,0,68.148148,1.000000,0.522907,0.059038,3,1,0,...,0,0,0,0,0,0,0,0,0,0
202217,23,16,0,100.000000,1.000000,0.522907,0.068591,3,1,0,...,0,1,0,0,30,13,14,36,0,16
35039,44,36,0,36.782616,0.906250,0.001502,0.050708,3,1,0,...,1,1,0,0,4,0,0,0,0,6
57443,25,18,0,100.000000,1.000000,0.003638,0.060428,2,1,0,...,0,1,0,0,14,4,15,72,0,14


In [19]:
y_train

125096    0
52769     0
202217    1
35039     0
57443     1
         ..
119879    1
103694    1
131932    0
146867    1
121958    1
Name: label, Length: 188636, dtype: int64

In [20]:
X_train.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)

y_train.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)

In [21]:
X_train.head()

,URLLength,DomainLength,IsDomainIP,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,...,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef
0,33,25,0,29.649226,0.772727,0.005977,0.056982,2,2,0,...,0,0,0,0,0,0,0,0,0,2
1,19,13,0,68.148148,1.000000,0.522907,0.059038,3,1,0,...,0,0,0,0,0,0,0,0,0,0
2,23,16,0,100.000000,1.000000,0.522907,0.068591,3,1,0,...,0,1,0,0,30,13,14,36,0,16
3,44,36,0,36.782616,0.906250,0.001502,0.050708,3,1,0,...,1,1,0,0,4,0,0,0,0,6
4,25,18,0,100.000000,1.000000,0.003638,0.060428,2,1,0,...,0,1,0,0,14,4,15,72,0,14


In [22]:
X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 188636 entries, 0 to 188635
Data columns (total 50 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   URLLength                   188636 non-null  int64  
 1   DomainLength                188636 non-null  int64  
 2   IsDomainIP                  188636 non-null  int64  
 3   URLSimilarityIndex          188636 non-null  float64
 4   CharContinuationRate        188636 non-null  float64
 5   TLDLegitimateProb           188636 non-null  float64
 6   URLCharProb                 188636 non-null  float64
 7   TLDLength                   188636 non-null  int64  
 8   NoOfSubDomain               188636 non-null  int64  
 9   HasObfuscation              188636 non-null  int64  
 10  NoOfObfuscatedChar          188636 non-null  int64  
 11  ObfuscationRatio            188636 non-null  float64
 12  NoOfLettersInURL            188636 non-null  int64  
 13  LetterRatioInURL         

### Outlier Detection 

In [23]:
binary_cols = []
continuous_cols = []


for col in X_train.columns:
    unique_val = set(X_train[col].dropna().unique())
    if unique_val.issubset({1,0,0.0,1.0}):
        binary_cols.append(col)
    else:
        continuous_cols.append(col)


print('Binary columns length:',len(binary_cols))
print('Continuous columns length:',len(continuous_cols))

Binary columns length: 19
Continuous columns length: 31


In [45]:
continuous_cols

['URLLength',
 'DomainLength',
 'URLSimilarityIndex',
 'CharContinuationRate',
 'TLDLegitimateProb',
 'URLCharProb',
 'TLDLength',
 'NoOfSubDomain',
 'NoOfObfuscatedChar',
 'ObfuscationRatio',
 'NoOfLettersInURL',
 'LetterRatioInURL',
 'NoOfDegitsInURL',
 'DegitRatioInURL',
 'NoOfEqualsInURL',
 'NoOfQMarkInURL',
 'NoOfAmpersandInURL',
 'NoOfOtherSpecialCharsInURL',
 'SpacialCharRatioInURL',
 'LineOfCode',
 'LargestLineLength',
 'DomainTitleMatchScore',
 'URLTitleMatchScore',
 'NoOfPopup',
 'NoOfiFrame',
 'NoOfImage',
 'NoOfCSS',
 'NoOfJS',
 'NoOfSelfRef',
 'NoOfEmptyRef',
 'NoOfExternalRef']

In [24]:
# IQR detection on continuous columns only 

outlier_report = []

for col in continuous_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outlier_count = ((X_train[col] < lower) | (X_train[col] > upper)).sum()
    outlier_pct = (outlier_count / len(X_train))*100

    outlier_report.append({
        'column': col,
        'outlier_count': outlier_count,
        'outlier_pct': round(outlier_pct, 2),
        'lower_bound': round(lower, 3),
        'upper': round(upper, 3),
        'min': round(X_train[col].min(), 3),
        'max': round(X_train[col].max(), 3) 
    })


report_df = pd.DataFrame(outlier_report).sort_values('outlier_pct', ascending=False)
print(report_df.to_string(index=False))

                    column  outlier_count  outlier_pct  lower_bound     upper    min          max
             NoOfSubDomain          45992        24.38        1.000     1.000  0.000       10.000
           NoOfDegitsInURL          41167        21.82        0.000     0.000  0.000     2011.000
           DegitRatioInURL          41167        21.82        0.000     0.000  0.000        0.678
              NoOfEmptyRef          31019        16.44       -1.500     2.500  0.000     4887.000
                NoOfiFrame          27560        14.61       -1.500     2.500  0.000     1602.000
                   NoOfCSS          18280         9.69      -10.500    17.500  0.000    35820.000
           NoOfExternalRef          18225         9.66      -83.000   141.000  0.000    27516.000
                 URLLength          17939         9.51        6.500    50.500 13.000     5794.000
          NoOfLettersInURL          15647         8.29       -5.000    35.000  0.000     4992.000
                Line

In [25]:
report_df

,column,outlier_count,outlier_pct,lower_bound,upper,min,max
7,NoOfSubDomain,45992,24.38,1.000,1.000,0.000,1.000000e+01
12,NoOfDegitsInURL,41167,21.82,0.000,0.000,0.000,2.011000e+03
13,DegitRatioInURL,41167,21.82,0.000,0.000,0.000,6.780000e-01
29,NoOfEmptyRef,31019,16.44,-1.500,2.500,0.000,4.887000e+03
24,NoOfiFrame,27560,14.61,-1.500,2.500,0.000,1.602000e+03
26,NoOfCSS,18280,9.69,-10.500,17.500,0.000,3.582000e+04
30,NoOfExternalRef,18225,9.66,-83.000,141.000,0.000,2.751600e+04
0,URLLength,17939,9.51,6.500,50.500,13.000,5.794000e+03
10,NoOfLettersInURL,15647,8.29,-5.000,35.000,0.000,4.992000e+03
19,LineOfCode,15387,8.16,-1870.500,3165.500,2.000,4.426660e+05


In [ ]:
# Handling the outliers 

for col in continuous_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)

    IQR = Q3 - Q1

    upper = Q3 + 1.5 * IQR
    X_train[col] = X_train[col].clip(upper=upper)
    X_test[col] = X_test[col].clip(upper=upper)


In [27]:
column_name = X_train.columns.to_list()

In [28]:
column_name

['URLLength',
 'DomainLength',
 'IsDomainIP',
 'URLSimilarityIndex',
 'CharContinuationRate',
 'TLDLegitimateProb',
 'URLCharProb',
 'TLDLength',
 'NoOfSubDomain',
 'HasObfuscation',
 'NoOfObfuscatedChar',
 'ObfuscationRatio',
 'NoOfLettersInURL',
 'LetterRatioInURL',
 'NoOfDegitsInURL',
 'DegitRatioInURL',
 'NoOfEqualsInURL',
 'NoOfQMarkInURL',
 'NoOfAmpersandInURL',
 'NoOfOtherSpecialCharsInURL',
 'SpacialCharRatioInURL',
 'IsHTTPS',
 'LineOfCode',
 'LargestLineLength',
 'HasTitle',
 'DomainTitleMatchScore',
 'URLTitleMatchScore',
 'HasFavicon',
 'Robots',
 'IsResponsive',
 'NoOfURLRedirect',
 'NoOfSelfRedirect',
 'HasDescription',
 'NoOfPopup',
 'NoOfiFrame',
 'HasExternalFormSubmit',
 'HasSocialNet',
 'HasSubmitButton',
 'HasHiddenFields',
 'HasPasswordField',
 'Bank',
 'Pay',
 'Crypto',
 'HasCopyrightInfo',
 'NoOfImage',
 'NoOfCSS',
 'NoOfJS',
 'NoOfSelfRef',
 'NoOfEmptyRef',
 'NoOfExternalRef']

### Feature Selection 

In [ ]:
# Step 1: Pairwise Correlation
corr_matrix = X_train.corr().abs()
upper_tri   = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = set()
for col in upper_tri.columns:
    for corr_feat in upper_tri.index[upper_tri[col] > 0.85].tolist():
        if col in to_drop or corr_feat in to_drop:
            continue
        if abs(X_train[col].corr(y_train)) >= abs(X_train[corr_feat].corr(y_train)):
            to_drop.add(corr_feat)
        else:
            to_drop.add(col)

X_train = X_train.drop(columns=list(to_drop))
X_test  = X_test.drop(columns=list(to_drop))

# Step 2: VIF
while True:
    X_reset    = X_train.reset_index(drop=True)
    vif_scores = pd.Series(
        [variance_inflation_factor(X_reset.values, i) for i in range(X_reset.shape[1])],
        index=X_train.columns
    )
    if vif_scores.max() > 5.0:
        drop_col = vif_scores.idxmax()
        X_train  = X_train.drop(columns=[drop_col])
        X_test   = X_test.drop(columns=[drop_col])
    else:
        break

print("Remaining features:", X_train.columns.tolist())

c:\anaconda3\envs\venv\Lib\site-packages\statsmodels\regression\linear_model.py:1784: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.uncentered_tss


Remaining features: ['IsDomainIP', 'TLDLegitimateProb', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'LargestLineLength', 'DomainTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef']


In [32]:
X_test.shape

(47159, 35)

In [33]:
X_train.shape 

(188636, 35)

In [34]:
X_train.head()

,IsDomainIP,TLDLegitimateProb,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,...,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef
0,0,0.005977,0,0,0.0,0,0.0,0,0,0,...,0,0,0,0,0,0.0,0.0,0,0.0,2
1,0,0.522907,0,0,0.0,0,0.0,0,0,0,...,0,0,0,0,0,0.0,0.0,0,0.0,0
2,0,0.522907,0,0,0.0,0,0.0,0,0,0,...,0,0,1,0,0,30.0,13.0,36,0.0,16
3,0,0.001502,0,0,0.0,0,0.0,0,0,0,...,0,1,1,0,0,4.0,0.0,0,0.0,6
4,0,0.003638,0,0,0.0,0,0.0,0,0,0,...,0,0,1,0,0,14.0,4.0,72,0.0,14


In [ ]:
# Cross-validation strategy
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

thresholds = [0.01, 0.02, 0.05, "mean", "median", 0.10]
results = {}

for thresh in thresholds:

    # Feature selector
    feature_selector = SelectFromModel(
        estimator=RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        ),
        threshold=thresh
    )

    # Final classifier
    classifier = RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    )

    # Pipeline
    pipeline = Pipeline([
        ('feature_selection', feature_selector),
        ('classification', classifier)
    ])

    # Cross-validation score
    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='accuracy',
        n_jobs=-1
    )

    # ---- Count selected features safely ----
    pipeline.fit(X_train, y_train)
    n_features = pipeline.named_steps['feature_selection'].transform(X_train).shape[1]

    # Store results
    results[str(thresh)] = {
        'mean_score': np.mean(scores),
        'std_score': np.std(scores),
        'n_features': n_features
    }

    print(
        f"Threshold: {str(thresh):>8} | "
        f"Features Selected: {n_features:>3} | "
        f"CV Accuracy: {np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )

Threshold:     0.01 | Features Selected:  14 | CV Accuracy: 0.9975 ± 0.0001
Threshold:     0.02 | Features Selected:  12 | CV Accuracy: 0.9974 ± 0.0002
Threshold:     0.05 | Features Selected:   7 | CV Accuracy: 0.9954 ± 0.0014
Threshold:     mean | Features Selected:  10 | CV Accuracy: 0.9973 ± 0.0002
Threshold:   median | Features Selected:  18 | CV Accuracy: 0.9976 ± 0.0001
Threshold:      0.1 | Features Selected:   3 | CV Accuracy: 0.9893 ± 0.0005


In [36]:
# Appling best threshold
selector = SelectFromModel(
    estimator=RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    threshold='median'
)
selector.fit(X_train, y_train)

# Transform
X_train_selected = pd.DataFrame(
    selector.transform(X_train),
    columns=X_train.columns[selector.get_support()]
)
X_test_selected = pd.DataFrame(
    selector.transform(X_test),
    columns=X_train.columns[selector.get_support()]
)

print("Selected features:", X_train_selected.columns.tolist())
print("Shape:", X_train_selected.shape)

Selected features: ['TLDLegitimateProb', 'NoOfOtherSpecialCharsInURL', 'LargestLineLength', 'DomainTitleMatchScore', 'HasFavicon', 'IsResponsive', 'HasDescription', 'NoOfiFrame', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'Pay', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef']
Shape: (188636, 18)


In [37]:
X_train_selected.head()

,TLDLegitimateProb,NoOfOtherSpecialCharsInURL,LargestLineLength,DomainTitleMatchScore,HasFavicon,IsResponsive,HasDescription,NoOfiFrame,HasSocialNet,HasSubmitButton,HasHiddenFields,Pay,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef
0,0.005977,3.0,869.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0
1,0.522907,1.0,28.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.522907,1.0,8022.0,75.000,0.0,1.0,1.0,2.5,1.0,0.0,1.0,1.0,0.0,30.0,13.0,36.0,0.0,16.0
3,0.001502,2.0,817.0,3.125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,4.0,0.0,0.0,0.0,6.0
4,0.003638,1.0,15263.0,100.000,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,14.0,4.0,72.0,0.0,14.0


In [ ]:
ARTIFACTS_DIR = "model/artifacts"
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

print(f"Saving artifacts to: {os.path.abspath(ARTIFACTS_DIR)}")

Saving artifacts to: C:\Users\AYAN SONI\Documents\All Projects\nullthreat\model\artifacts


In [54]:
X_train_selected.to_csv(f"{ARTIFACTS_DIR}/X_train_selected.csv", index=False)
X_test_selected.to_csv(f"{ARTIFACTS_DIR}/X_test_selected.csv", index=False)

y_train.to_csv(f"{ARTIFACTS_DIR}/y_train.csv", index=False)
y_test.to_csv(f"{ARTIFACTS_DIR}/y_test.csv", index=False)

print(f"Saved train shape : {X_train_selected.shape}")
print(f"Saved test shape : {X_test_selected.shape}")

Saved train shape : (188636, 18)
Saved test shape : (47159, 18)


In [55]:
selected_features = X_train_selected.columns.tolist()

with open(f"{ARTIFACTS_DIR}/selected_features.json", "w") as f:
    json.dump(selected_features, f, indent=2)

print(f"Saved {len(selected_features)} selected features")
print(selected_features)

Saved 18 selected features
['TLDLegitimateProb', 'NoOfOtherSpecialCharsInURL', 'LargestLineLength', 'DomainTitleMatchScore', 'HasFavicon', 'IsResponsive', 'HasDescription', 'NoOfiFrame', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'Pay', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef']


In [56]:
outlier_caps = {}
for col in continuous_cols:
    if col not in X_train.columns:
        continue

    Q1  = X_train[col].quantile(0.25)
    Q3  = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    outlier_caps[col] = round(Q3 + 1.5 * IQR, 6)

with open(f"{ARTIFACTS_DIR}/outlier_caps.json", "w") as f:
    json.dump(outlier_caps, f, indent=2)

print(f"Saved outlier caps for {len(outlier_caps)} continuous columns")
print(list(outlier_caps.keys()))

Saved outlier caps for 18 continuous columns
['TLDLegitimateProb', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'LargestLineLength', 'DomainTitleMatchScore', 'NoOfPopup', 'NoOfiFrame', 'NoOfImage', 'NoOfCSS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef']


In [57]:
joblib.dump(selector, f"{ARTIFACTS_DIR}/rf_selector.pkl")

print(f"Saved RF selector  →  {ARTIFACTS_DIR}/rf_selector.pkl")

Saved RF selector  →  model/artifacts/rf_selector.pkl


In [62]:
print("=" * 50)
print("   PREPROCESSING COMPLETE")
print("=" * 50)
print(f"  Original features    : {phiusiil_df.shape[1]}")
print(f"  After corr/VIF drop  : {X_train.shape[1]}")
print(f"  Final selected       : {X_train_selected.shape[1]}")
print(f"  Train samples        : {X_train_selected.shape[0]}")
print(f"  Test  samples        : {X_test_selected.shape[0]}")
print(f"  Artifacts saved to   : {ARTIFACTS_DIR}")
print("=" * 50)

   PREPROCESSING COMPLETE
  Original features    : 51
  After corr/VIF drop  : 35
  Final selected       : 18
  Train samples        : 188636
  Test  samples        : 47159
  Artifacts saved to   : model/artifacts
